In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys

sys.path.append("../src")

import jax
import jax.numpy as jnp

import numpy as np

from onlineEM import em_config
from onlineEM.hd import hdstm
from onlineEM.sd import stm

from scipy.stats import multivariate_t
from torch.utils import data


jax.config.update("jax_enable_x64", False)

In [3]:
def generate_mixture_of_student(num_samples, means, covariances, nus, weights):
    num_components = len(means)
    component_indices = np.random.choice(num_components, size=num_samples, p=weights)

    samples = np.zeros((num_samples, len(means[0])))

    for i in range(num_components):
        indices = component_indices == i
        num_samples_component = np.sum(indices)
        component_samples = multivariate_t.rvs(
            loc=means[i], shape=covariances[i], df=nus[i], size=num_samples_component
        )
        samples[indices] = component_samples

    return samples


class XDataset(data.Dataset):
    def __init__(self, X):
        super().__init__()
        self.data = X

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def numpy_collate(batch):
    if isinstance(batch[0], np.ndarray):
        return np.stack(batch)
    elif isinstance(batch[0], (tuple, list)):
        transposed = zip(*batch)
        return [numpy_collate(samples) for samples in transposed]
    else:
        return np.array(batch)

In [4]:
mean1 = np.array([1, 2, 3, 4, 5])
covariance1 = np.diag([5, 5, 0.05, 0.05, 0.05])
nu1 = 5.0
covariance1 = np.diag([5, 5, 0.05, 0.05, 0.05])
weight1 = 0.4

mean2 = np.array([5, 4, 3, 2, 1])
covariance2 = np.diag([3, 3, 3, 0.03, 0.03])
nu2 = 5.0
weight2 = 0.3

mean3 = np.array([0, 0, 0, 0, 0])
covariance3 = np.diag([1, 1, 0.01, 0.01, 0.01])
covariance3 = np.diag([1, 1, 0.01, 0.01, 0.01])
nu3 = 5.0
weight3 = 0.3


means = [mean1, mean2, mean3]
covariances = [covariance1, covariance2, covariance3]
weights = [weight1, weight2, weight3]
nus = [nu1, nu2, nu3]

num_samples = 100000
X = generate_mixture_of_student(num_samples, means, covariances, nus, weights)
X = jnp.array(X)

X_dataset = XDataset(X)
X = data.DataLoader(
    X_dataset, batch_size=256, shuffle=True, collate_fn=numpy_collate, drop_last=True
)

# Test STM

In [5]:
config = {
    "n_components": 3,
    "num_features": 5,
    "num_epochs": 3,
    "mini_batch_size": 256,
}

config = em_config(**config)

model_hd = hdstm()
model_sd = stm()

In [ ]:
# SD model
config_sd, params_sd, stats_sd = model_sd.burnin(X, config)
params_sd, stats_sd = model_sd.online_epochs(X, params_sd, stats_sd, config_sd)

# HD model
config_hd, params_hd, stats_hd = model_hd.burnin(X, config)
params_hd, stats_hd = model_hd.online_epochs(X, params_hd, stats_hd, config_hd)